# Hyperparameter Optimization

## Scientific objective
Run fixed-budget Optuna optimization using training/validation scaffold partitions only, with objective and trial provenance persisted.

## Inputs
- Morgan features
- Primary split
- model search spaces

## Expected outputs
- `results/metrics/optuna_trials.csv`
- `models/qsar/optimized_*.joblib`
- study metadata

## Dependencies
Optuna, scikit-learn

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
The test partition is never exposed to Optuna. Trial budgets are fixed by execution profile.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Optimization variance can exceed small metric differences; repeated scaffold splits remain necessary.

## Next notebook
[16_calibration_and_uncertainty.ipynb](./16_calibration_and_uncertainty.ipynb)

In [3]:
from pathlib import Path
import os

# Apply thread restrictions before importing NumPy/scikit-learn.
for variable in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "BLIS_NUM_THREADS",
]:
    os.environ.setdefault(variable, "1")

os.environ.setdefault("PYTHONHASHSEED", "20260723")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import random
import warnings

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError(
        "Run this notebook from the repository root or notebooks directory"
    )

os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])

# Classical-only seeding: do not initialize PyTorch in this notebook.
random.seed(SEED)
np.random.seed(SEED)

print(
    {
        "root": str(ROOT),
        "profile": PROFILE,
        "seed": SEED,
        "sample_cap": PROFILE_CONFIG["sample_cap_per_endpoint"],
        "optuna_trials": PROFILE_CONFIG["optuna_trials"],
    }
)

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723, 'sample_cap': None, 'optuna_trials': 50}


In [4]:
import warnings

# Suppress the harmless missing Jupyter-widget warning.
warnings.filterwarnings(
    "ignore",
    message="IProgress not found.*",
)

import joblib
import optuna

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler

from toxicity_screening.evaluation import (
    SafeLogisticPipeline,
    safe_logistic_positive_probability,
)


def build_safe_logistic_model(c_value: float) -> SafeLogisticPipeline:
    """Construct a single-threaded, BLAS-safe logistic pipeline."""
    return SafeLogisticPipeline(
        [
            (
                "scale",
                StandardScaler(with_mean=False),
            ),
            (
                "model",
                LogisticRegression(
                    C=float(c_value),
                    solver="liblinear",
                    penalty="l2",
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=SEED,
                ),
            ),
        ]
    )


archive = np.load(
    ROOT / "data/processed/morgan_features.npz",
    allow_pickle=False,
)

X = archive["X"]
ids = archive["molecule_id"].astype(str)

index = pd.DataFrame(
    {
        "molecule_id": ids,
        "row": np.arange(len(ids), dtype=np.int64),
    }
)

records = pd.read_parquet(
    ROOT / "data/processed/modeling_records.parquet"
).merge(
    index,
    on="molecule_id",
    validate="many_to_one",
)

if len(ids) != X.shape[0]:
    raise ValueError(
        f"Feature/identifier mismatch: {X.shape[0]} rows versus {len(ids)} IDs"
    )

sample_cap = PROFILE_CONFIG["sample_cap_per_endpoint"]
trial_budget = int(PROFILE_CONFIG["optuna_trials"])

trials = []

for endpoint, frame in records.loc[
    records["label"].notna()
].groupby("endpoint", sort=False):

    train = frame.loc[
        frame["scaffold_split"] == "train"
    ].copy()

    validation = frame.loc[
        frame["scaffold_split"] == "validation"
    ].copy()

    if sample_cap and len(train) > int(sample_cap):
        train = train.sample(
            n=int(sample_cap),
            random_state=SEED,
        )

    if train.empty or validation.empty:
        raise ValueError(
            f"{endpoint}: empty training or validation partition"
        )

    y_train = train["label"].astype(int).to_numpy()
    y_validation = validation["label"].astype(int).to_numpy()

    if np.unique(y_train).size < 2:
        raise ValueError(
            f"{endpoint}: training partition contains only one class"
        )

    if np.unique(y_validation).size < 2:
        raise ValueError(
            f"{endpoint}: validation partition contains only one class"
        )

    x_train = X[train["row"].astype(int).to_numpy()]
    x_validation = X[
        validation["row"].astype(int).to_numpy()
    ]

    print(
        f"[OPTUNA] endpoint_started "
        f"endpoint={endpoint} "
        f"train_rows={len(train)} "
        f"validation_rows={len(validation)} "
        f"trials={trial_budget}",
        flush=True,
    )

    def objective(trial: optuna.Trial) -> float:
        c_value = trial.suggest_float(
            "C",
            1e-3,
            1e2,
            log=True,
        )

        print(
            f"[OPTUNA] trial_started "
            f"endpoint={endpoint} "
            f"trial={trial.number} "
            f"C={c_value:.8g}",
            flush=True,
        )

        model = build_safe_logistic_model(c_value)
        model.fit(x_train, y_train)

        probability = safe_logistic_positive_probability(
            model,
            x_validation,
            batch_size=256,
        )

        score = float(
            average_precision_score(
                y_validation,
                probability,
            )
        )

        print(
            f"[OPTUNA] trial_completed "
            f"endpoint={endpoint} "
            f"trial={trial.number} "
            f"pr_auc={score:.6f}",
            flush=True,
        )

        return score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=SEED,
        ),
    )

    study.optimize(
        objective,
        n_trials=trial_budget,
        n_jobs=1,
        show_progress_bar=False,
        gc_after_trial=True,
    )

    best_model = build_safe_logistic_model(
        study.best_params["C"]
    )
    best_model.fit(x_train, y_train)

    model_path = (
        ROOT
        / "models"
        / "qsar"
        / f"optimized_{endpoint}.joblib"
    )
    model_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    joblib.dump(best_model, model_path)

    for completed_trial in study.trials:
        trials.append(
            {
                "endpoint": endpoint,
                "number": completed_trial.number,
                "value": completed_trial.value,
                "state": str(completed_trial.state),
                **completed_trial.params,
            }
        )

    print(
        f"[OPTUNA] endpoint_completed "
        f"endpoint={endpoint} "
        f"best_C={study.best_params['C']:.8g} "
        f"best_pr_auc={study.best_value:.6f} "
        f"model={model_path}",
        flush=True,
    )

trial_frame = pd.DataFrame(trials)

output_path = (
    ROOT
    / "results"
    / "metrics"
    / "optuna_trials.csv"
)
output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)
trial_frame.to_csv(
    output_path,
    index=False,
)

print(
    f"[OPTUNA] completed "
    f"endpoints={trial_frame['endpoint'].nunique()} "
    f"trials={len(trial_frame)} "
    f"output={output_path}",
    flush=True,
)

display(trial_frame)

[OPTUNA] endpoint_started endpoint=herg_blockade train_rows=8971 validation_rows=2095 trials=50


[I 2026-08-06 15:45:25,562] A new study created in memory with name: no-name-8b9482ae-8c6b-4130-991c-f2b5009a3ad2


[OPTUNA] trial_started endpoint=herg_blockade trial=0 C=2.6890789
[OPTUNA] trial_completed endpoint=herg_blockade trial=0 pr_auc=0.762309


[I 2026-08-06 15:45:28,198] Trial 0 finished with value: 0.7623088436346327 and parameters: {'C': 2.6890788551555658}. Best is trial 0 with value: 0.7623088436346327.


[OPTUNA] trial_started endpoint=herg_blockade trial=1 C=0.18532294
[OPTUNA] trial_completed endpoint=herg_blockade trial=1 pr_auc=0.772925


...

[I 2026-08-06 15:49:22,139] Trial 48 finished with value: 0.575449298413485 and parameters: {'C': 0.0011509843264870278}. Best is trial 23 with value: 0.5805780554212676.


[OPTUNA] trial_started endpoint=SR-MMP trial=49 C=0.0010120457
[OPTUNA] trial_completed endpoint=SR-MMP trial=49 pr_auc=0.580564


[I 2026-08-06 15:49:22,703] Trial 49 finished with value: 0.5805640716034428 and parameters: {'C': 0.0010120456500930422}. Best is trial 23 with value: 0.5805780554212676.


[OPTUNA] endpoint_completed endpoint=SR-MMP best_C=0.0010096242 best_pr_auc=0.580578 model=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\models\qsar\optimized_SR-MMP.joblib
[OPTUNA] completed endpoints=6 trials=300 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\results\metrics\optuna_trials.csv


,endpoint,number,value,state,C
0,herg_blockade,0,0.762309,1,2.689079
1,herg_blockade,1,0.772925,1,0.185323
2,herg_blockade,2,0.764606,1,0.974238
3,herg_blockade,3,0.774002,1,0.160387
4,herg_blockade,4,0.826001,1,0.003083
...,...,...,...,...,...
295,SR-MMP,45,0.570293,1,0.001562
296,SR-MMP,46,0.459791,1,0.458987
297,SR-MMP,47,0.558946,1,0.002667
298,SR-MMP,48,0.575449,1,0.001151


### Completion gate
Confirm that the declared artifacts exist before continuing to `16_calibration_and_uncertainty.ipynb`.